In [7]:
#!/usr/bin/env python3
"""
TIMS SWITRS Query API client (querySwitrsNew.php) + JSON export

Folder behavior (UPDATED):
- Output folder is NOT based on query time.
- Folder name: <start_date>_to_<end_date>_<county>
- If it already exists: DO NOT create a new folder; just overwrite files inside it.
- No case_ids.json is produced.

Output:
  out_tims_json/<start_date>_to_<end_date>_<county>/
    - manifest.json
    - response_full.json  (or response_chunk_XXXX.json if chunked)
"""

import json
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests


# ----------------------------
# County list (names only) — NO "ALL"
# ----------------------------
COUNTIES = [
    "ALAMEDA","ALPINE","AMADOR","BUTTE","CALAVERAS","COLUSA","CONTRA COSTA","DEL NORTE",
    "EL DORADO","FRESNO","GLENN","HUMBOLDT","IMPERIAL","INYO","KERN","KINGS","LAKE","LASSEN",
    "LOS ANGELES","MADERA","MARIN","MARIPOSA","MENDOCINO","MERCED","MODOC","MONO","MONTEREY","NAPA",
    "NEVADA","ORANGE","PLACER","PLUMAS","RIVERSIDE","SACRAMENTO","SAN BENITO","SAN BERNARDINO",
    "SAN DIEGO","SAN FRANCISCO","SAN JOAQUIN","SAN LUIS OBISPO","SAN MATEO","SANTA BARBARA",
    "SANTA CLARA","SANTA CRUZ","SHASTA","SIERRA","SISKIYOU","SOLANO","SONOMA","STANISLAUS","SUTTER",
    "TEHAMA","TRINITY","TULARE","TUOLUMNE","VENTURA","YOLO","YUBA"
]


# ----------------------------
# Config
# ----------------------------
OUT_ROOT = Path("out_tims_json")
CHUNK_SIZE = 50_000  # if response list is larger than this, export chunks


# ----------------------------
# Helpers
# ----------------------------
def normalize_county(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().upper())

def validate_date_yyyy_mm_dd(s: str) -> str:
    s = s.strip()
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        raise ValueError(f"Date must be YYYY-MM-DD, got: {s}")
    return s

def build_base_query(start_date: str, end_date: str, county: str) -> str:
    return (
        f"COLLISION_DATE >= '{start_date}' "
        f"AND COLLISION_DATE <= '{end_date}' "
        f"AND S.COUNTY = '{county}'"
    )

def safe_tag(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[^A-Za-z0-9._-]+", "-", s)
    return s.strip("-")

def print_county_options(counties: List[str], cols: int = 4) -> None:
    width = max(len(c) for c in counties) + 6
    for i in range(0, len(counties), cols):
        row = counties[i:i+cols]
        parts = []
        for j, name in enumerate(row, start=i+1):
            parts.append(f"{j:2d}) {name}".ljust(width))
        print("".join(parts))

def prompt_county() -> str:
    print("\nChoose a COUNTY (no ALL). Options:")
    print("-" * 72)
    print_county_options(COUNTIES, cols=4)
    print("-" * 72)

    while True:
        raw = input("Enter county number (e.g., 1) or name (e.g., ALAMEDA): ").strip()
        if not raw:
            print("[ERROR] Please enter a value.")
            continue

        if raw.isdigit():
            idx = int(raw)
            if 1 <= idx <= len(COUNTIES):
                return COUNTIES[idx - 1]
            print(f"[ERROR] Number must be between 1 and {len(COUNTIES)}.")
            continue

        name = normalize_county(raw)
        if name in COUNTIES:
            return name

        print("[ERROR] County not recognized. Please choose from the list above.")

def make_output_dir_overwrite(out_root: Path, start_date: str, end_date: str, county: str) -> Path:
    """
    Folder is deterministic and we always reuse it.
    - Creates it if missing
    - If exists: reuse/overwrite files (no new folder)
    """
    base = safe_tag(f"{start_date}_to_{end_date}_{county}")
    out_root.mkdir(parents=True, exist_ok=True)
    out_dir = out_root / base
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

def find_primary_list_container(payload: Any) -> Optional[str]:
    if not isinstance(payload, dict):
        return None
    for key in ("data", "results", "rows"):
        if key in payload and isinstance(payload[key], list):
            return key
    return None

def dump_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def delete_old_chunks(out_dir: Path) -> None:
    """
    If previous run created response_chunk_*.json, remove them so we don't leave stale files.
    """
    for p in out_dir.glob("response_chunk_*.json"):
        try:
            p.unlink()
        except Exception:
            pass

def export_payload_json(out_dir: Path, payload: Any) -> Dict[str, Any]:
    """
    Export payload to JSON:
      - If payload is a big list -> chunk it
      - If payload is dict with a big list under ('data'/'results'/'rows') -> chunk that list
      - Else -> write response_full.json

    Overwrite behavior:
      - Removes any stale response_chunk_*.json before writing.
      - If writing response_full.json, overwrites it.
    """
    delete_old_chunks(out_dir)

    export_info: Dict[str, Any] = {"mode": None, "files": []}

    # Case A: payload is a list
    if isinstance(payload, list):
        n = len(payload)
        if n > CHUNK_SIZE:
            export_info["mode"] = "chunked_list"
            num_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
            for i in range(num_chunks):
                chunk = payload[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]
                fn = f"response_chunk_{i+1:04d}.json"
                dump_json(out_dir / fn, chunk)
                export_info["files"].append(fn)
            export_info["total_rows"] = n
            export_info["chunk_size"] = CHUNK_SIZE
        else:
            export_info["mode"] = "single"
            fn = "response_full.json"
            dump_json(out_dir / fn, payload)
            export_info["files"].append(fn)
            export_info["total_rows"] = n
        return export_info

    # Case B: payload is a dict that contains a primary list
    primary_key = find_primary_list_container(payload)
    if primary_key is not None:
        rows = payload[primary_key]
        n = len(rows)
        if n > CHUNK_SIZE:
            export_info["mode"] = f"chunked_dict[{primary_key}]"
            num_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
            for i in range(num_chunks):
                chunk_rows = rows[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]
                chunk_payload = dict(payload)
                chunk_payload[primary_key] = chunk_rows
                fn = f"response_chunk_{i+1:04d}.json"
                dump_json(out_dir / fn, chunk_payload)
                export_info["files"].append(fn)
            export_info["total_rows"] = n
            export_info["chunk_size"] = CHUNK_SIZE
            export_info["container_key"] = primary_key
        else:
            export_info["mode"] = "single"
            fn = "response_full.json"
            dump_json(out_dir / fn, payload)
            export_info["files"].append(fn)
            export_info["total_rows"] = n
            export_info["container_key"] = primary_key
        return export_info

    # Case C: payload is dict/other without a big list: single file
    export_info["mode"] = "single"
    fn = "response_full.json"
    dump_json(out_dir / fn, payload)
    export_info["files"].append(fn)
    return export_info


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    print("TIMS SWITRS Query (querySwitrsNew.php) + JSON export")
    print("-" * 72)

    start_date = validate_date_yyyy_mm_dd(input("Enter COLLISION start date (YYYY-MM-DD): "))
    end_date   = validate_date_yyyy_mm_dd(input("Enter COLLISION end date   (YYYY-MM-DD): "))
    county     = prompt_county()

    base_query = build_base_query(start_date, end_date, county)

    url = "https://tims.berkeley.edu/tools/query/querySwitrsNew.php"

    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Origin": "https://tims.berkeley.edu",
        "Referer": "https://tims.berkeley.edu/tools/query/summary.php",
        "X-Requested-With": "XMLHttpRequest",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome Safari",
    }

    data = {"baseQuery": base_query, "pedorbike": ""}

    print("\n[INFO] baseQuery:")
    print(base_query)
    print("\n[INFO] POSTing to:", url)

    with requests.Session() as sess:
        resp = sess.post(url, headers=headers, data=data, timeout=120)
        resp.raise_for_status()
        try:
            payload = resp.json()
        except Exception:
            OUT_ROOT.mkdir(parents=True, exist_ok=True)
            raw_path = OUT_ROOT / "response_raw_text.txt"
            raw_path.write_text(resp.text, encoding="utf-8")
            print(f"\n[ERROR] Response was not valid JSON. Saved raw text to: {raw_path}")
            raise SystemExit(3)

    # Deterministic folder, reuse it, overwrite contents
    out_dir = make_output_dir_overwrite(OUT_ROOT, start_date, end_date, county)

    # Export response JSON (single or chunked) — overwriting previous files
    export_info = export_payload_json(out_dir, payload)

    # Manifest (includes save time inside, but folder name is deterministic)
    manifest = {
        "endpoint": url,
        "request": {
            "start_date": start_date,
            "end_date": end_date,
            "county": county,
            "baseQuery": base_query,
            "pedorbike": "",
        },
        "export": export_info,
        "saved_at_localtime": datetime.now().isoformat(timespec="seconds"),
    }
    dump_json(out_dir / "manifest.json", manifest)

    print("\n[SAVED] Output directory:")
    print(" ", out_dir.resolve())
    print("\n[SAVED] Files:")
    for fn in ["manifest.json", *export_info.get("files", [])]:
        print("  -", fn)


if __name__ == "__main__":
    main()

TIMS SWITRS Query (querySwitrsNew.php) + JSON export
------------------------------------------------------------------------

Choose a COUNTY (no ALL). Options:
------------------------------------------------------------------------
 1) ALAMEDA           2) ALPINE            3) AMADOR            4) BUTTE            
 5) CALAVERAS         6) COLUSA            7) CONTRA COSTA      8) DEL NORTE        
 9) EL DORADO        10) FRESNO           11) GLENN            12) HUMBOLDT         
13) IMPERIAL         14) INYO             15) KERN             16) KINGS            
17) LAKE             18) LASSEN           19) LOS ANGELES      20) MADERA           
21) MARIN            22) MARIPOSA         23) MENDOCINO        24) MERCED           
25) MODOC            26) MONO             27) MONTEREY         28) NAPA             
29) NEVADA           30) ORANGE           31) PLACER           32) PLUMAS           
33) RIVERSIDE        34) SACRAMENTO       35) SAN BENITO       36) SAN BERNARDINO   


In [25]:
#!/usr/bin/env python3
"""
TIMS SWITRS pipeline (2-step)
1) Query collisions (querySwitrsNew.php) -> saves response_full.json (+ manifest.json)
2) Read response_full.json, extract case IDs from payload["data"] (first column),
   normalize IDs (e.g., 228939735 -> 8939735; strip leading zeros),
   then download SWITRS CSV bundle(s) (download.php) in batches.

Output (NO ZIP outputs by default):
  out_tims_json/<start_date>_to_<end_date>_<county>/
    - response_full.json
    - manifest.json
    - csv/                    # final CSV tables (appended across batches)
        <table1>.csv
        <table2>.csv
        ...
    - parquet/ (optional)      # if WRITE_PARQUET=True
        <table_name>/batch_0001_part_0001.parquet, ...
    - downloads/ (optional)    # per-batch ID lists and (optionally) raw responses

Folder behavior:
- Folder name is deterministic: <start>_to_<end>_<county>
- If it already exists: REUSE it and OVERWRITE outputs (folder not recreated)

Requirements:
  pip install requests
Optional for parquet:
  pip install pandas pyarrow

Run:
  python tims_switrs_full_pipeline.py
"""

from __future__ import annotations

import json
import re
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests


# =============================================================================
# USER CONFIG
# =============================================================================
OUT_ROOT = Path("out_tims_json")

# Query response saving (usually dict; chunking is rarely needed)
QUERY_CHUNK_SIZE = 50_000  # only used if payload is a huge list

# Download batching (smaller is safer if the server truncates silently)
DOWNLOAD_BATCH_SIZE = 500      # try 200-1000 if you see issues
DOWNLOAD_TIMEOUT_SEC = 300

# What to output
KEEP_RAW_DOWNLOADS = False     # if True, keeps raw zip/csv/html from server in downloads/
WRITE_PARQUET = False          # if True, writes parquet parts under parquet/ (needs pandas+pyarrow)
PARQUET_CSV_CHUNKSIZE = 200_000

# Overwrite semantics inside existing folder
CLEAR_PREVIOUS_RUN_ARTIFACTS = True  # clears csv/, parquet/, downloads/, temp extract each run


# =============================================================================
# County list (NO "ALL")
# =============================================================================
COUNTIES = [
    "ALAMEDA","ALPINE","AMADOR","BUTTE","CALAVERAS","COLUSA","CONTRA COSTA","DEL NORTE",
    "EL DORADO","FRESNO","GLENN","HUMBOLDT","IMPERIAL","INYO","KERN","KINGS","LAKE","LASSEN",
    "LOS ANGELES","MADERA","MARIN","MARIPOSA","MENDOCINO","MERCED","MODOC","MONO","MONTEREY","NAPA",
    "NEVADA","ORANGE","PLACER","PLUMAS","RIVERSIDE","SACRAMENTO","SAN BENITO","SAN BERNARDINO",
    "SAN DIEGO","SAN FRANCISCO","SAN JOAQUIN","SAN LUIS OBISPO","SAN MATEO","SANTA BARBARA",
    "SANTA CLARA","SANTA CRUZ","SHASTA","SIERRA","SISKIYOU","SOLANO","SONOMA","STANISLAUS","SUTTER",
    "TEHAMA","TRINITY","TULARE","TUOLUMNE","VENTURA","YOLO","YUBA"
]


# =============================================================================
# Helpers: input / formatting
# =============================================================================
def normalize_county(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().upper())

def validate_date_yyyy_mm_dd(s: str) -> str:
    s = s.strip()
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        raise ValueError(f"Date must be YYYY-MM-DD, got: {s}")
    return s

def build_base_query(start_date: str, end_date: str, county: str) -> str:
    return (
        f"COLLISION_DATE >= '{start_date}' "
        f"AND COLLISION_DATE <= '{end_date}' "
        f"AND S.COUNTY = '{county}'"
    )

def safe_tag(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[^A-Za-z0-9._-]+", "-", s)
    return s.strip("-")

def print_county_options(counties: List[str], cols: int = 4) -> None:
    width = max(len(c) for c in counties) + 6
    for i in range(0, len(counties), cols):
        row = counties[i:i + cols]
        parts = []
        for j, name in enumerate(row, start=i + 1):
            parts.append(f"{j:2d}) {name}".ljust(width))
        print("".join(parts))

def prompt_county() -> str:
    print("\nChoose a COUNTY (no ALL). Options:")
    print("-" * 72)
    print_county_options(COUNTIES, cols=4)
    print("-" * 72)

    while True:
        raw = input("Enter county number (e.g., 1) or name (e.g., ALAMEDA): ").strip()
        if not raw:
            print("[ERROR] Please enter a value.")
            continue

        if raw.isdigit():
            idx = int(raw)
            if 1 <= idx <= len(COUNTIES):
                return COUNTIES[idx - 1]
            print(f"[ERROR] Number must be between 1 and {len(COUNTIES)}.")
            continue

        name = normalize_county(raw)
        if name in COUNTIES:
            return name

        print("[ERROR] County not recognized. Please choose from the list above.")


# =============================================================================
# Helpers: filesystem
# =============================================================================
def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def dump_json(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def make_output_dir_overwrite(out_root: Path, start_date: str, end_date: str, county: str) -> Path:
    """
    Deterministic folder, reuse it; overwrite contents.
    """
    out_root.mkdir(parents=True, exist_ok=True)
    out_dir = out_root / safe_tag(f"{start_date}_to_{end_date}_{county}")
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

def clear_run_artifacts(out_dir: Path) -> None:
    """
    Clears subfolders and known files, but does NOT delete/recreate the main folder.
    """
    # Remove subfolders
    for sub in ["csv", "parquet", "downloads", "_tmp_extract"]:
        p = out_dir / sub
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)

    # Remove known files (safe)
    for fn in ["response_full.json", "manifest.json", "response_raw_text.txt"]:
        p = out_dir / fn
        if p.exists():
            try:
                p.unlink()
            except Exception:
                pass

def looks_like_zip(content: bytes) -> bool:
    return len(content) >= 4 and content[:2] == b"PK"  # ZIP magic

def append_csv_skip_header(src_csv: Path, dst_csv: Path) -> None:
    """
    Append src_csv to dst_csv (streaming).
    If dst doesn't exist: copy whole file.
    If dst exists: append everything except first line (header).
    """
    ensure_dir(dst_csv.parent)
    if not dst_csv.exists():
        shutil.copyfile(src_csv, dst_csv)
        return

    with src_csv.open("r", encoding="utf-8", errors="replace") as fin, \
         dst_csv.open("a", encoding="utf-8", errors="replace") as fout:
        _ = fin.readline()  # skip header
        shutil.copyfileobj(fin, fout)


# =============================================================================
# Step 1: Query + save response_full.json
# =============================================================================
def find_primary_list_container(payload: Any) -> Optional[str]:
    if not isinstance(payload, dict):
        return None
    for key in ("data", "results", "rows"):
        if key in payload and isinstance(payload[key], list):
            return key
    return None

def export_query_payload(out_dir: Path, payload: Any) -> Dict[str, Any]:
    """
    Save query payload as response_full.json (or chunked).
    """
    # remove stale chunks from earlier runs
    for p in out_dir.glob("response_chunk_*.json"):
        try:
            p.unlink()
        except Exception:
            pass

    export_info: Dict[str, Any] = {"mode": None, "files": []}

    if isinstance(payload, list):
        n = len(payload)
        if n > QUERY_CHUNK_SIZE:
            export_info["mode"] = "chunked_list"
            num_chunks = (n + QUERY_CHUNK_SIZE - 1) // QUERY_CHUNK_SIZE
            for i in range(num_chunks):
                chunk = payload[i * QUERY_CHUNK_SIZE : (i + 1) * QUERY_CHUNK_SIZE]
                fn = f"response_chunk_{i+1:04d}.json"
                dump_json(out_dir / fn, chunk)
                export_info["files"].append(fn)
            export_info["total_rows"] = n
            export_info["chunk_size"] = QUERY_CHUNK_SIZE
        else:
            export_info["mode"] = "single"
            dump_json(out_dir / "response_full.json", payload)
            export_info["files"].append("response_full.json")
            export_info["total_rows"] = n
        return export_info

    # dict case: most common
    primary = find_primary_list_container(payload)
    if primary is not None and isinstance(payload.get(primary), list):
        export_info["container_key"] = primary
        export_info["total_rows"] = len(payload[primary])

    export_info["mode"] = "single"
    dump_json(out_dir / "response_full.json", payload)
    export_info["files"].append("response_full.json")
    return export_info

def run_query_and_save(
    sess: requests.Session,
    out_dir: Path,
    start_date: str,
    end_date: str,
    county: str,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Executes querySwitrsNew.php and saves response_full.json.
    Returns (payload_dict, export_info).
    """
    url = "https://tims.berkeley.edu/tools/query/querySwitrsNew.php"
    base_query = build_base_query(start_date, end_date, county)
    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Origin": "https://tims.berkeley.edu",
        "Referer": "https://tims.berkeley.edu/tools/query/summary.php",
        "X-Requested-With": "XMLHttpRequest",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome Safari",
    }
    data = {"baseQuery": base_query, "pedorbike": ""}

    print("\n[INFO] POSTing query to:", url)
    print("[INFO] baseQuery:", base_query)

    resp = sess.post(url, headers=headers, data=data, timeout=120)
    resp.raise_for_status()

    try:
        payload = resp.json()
    except Exception:
        raw_path = out_dir / "response_raw_text.txt"
        raw_path.write_text(resp.text, encoding="utf-8")
        raise RuntimeError(f"Query response not valid JSON. Saved raw text to: {raw_path}")

    export_info = export_query_payload(out_dir, payload)
    return payload, export_info


# =============================================================================
# Step 2: Extract case IDs from response_full.json["data"] first column
# =============================================================================
# def normalize_case_id_for_download(raw_id: str) -> Optional[str]:
#     """
#     Normalization rules:
#       - if digits:
#           * if len==9 and startswith '22' and remaining is 7 digits: strip '22'
#             example: '228939735' -> '8939735'
#           * strip leading zeros by int conversion
#       - return None if invalid
#     """
#     s = str(raw_id).strip().replace("\ufeff", "")
#     if not s or not s.isdigit():
#         return None

#     if len(s) == 9 and s.startswith("22") and len(s[2:]) == 7 and s[2:].isdigit():
#         s = s[2:]

#     try:
#         s = str(int(s))  # strips leading zeros
#     except Exception:
#         pass

#     if not s or s == "0":
#         return None
#     return s

import re
from typing import Optional

def normalize_case_id_for_download(raw_id: str) -> Optional[str]:
    """
    Keep case IDs EXACTLY as in response_full.json, except:
      - if it matches ^22(\d{7})$ then drop leading '22'
        e.g., 228939735 -> 8939735

    Do NOT int() cast (do NOT strip leading zeros).
    """
    s = str(raw_id).strip().replace("\ufeff", "")
    if not s.isdigit():
        return None

    m = re.fullmatch(r"22(\d{7})", s)
    if m:
        return m.group(1)

    return s

def extract_case_ids_from_response_full_json(path: Path) -> List[str]:
    """
    Reads response_full.json and extracts case ids from payload["data"]:
      - If data is dict: keys are case ids
      - If data is list: each row's first element row[0] is case id
      - If row is dict: tries common id keys
    Returns unique normalized IDs.
    """
    payload = json.loads(path.read_text(encoding="utf-8"))

    if not isinstance(payload, dict) or "data" not in payload:
        raise ValueError("response_full.json does not have top-level 'data' key (or not a dict).")

    data = payload["data"]

    raw_ids: List[str] = []

    if isinstance(data, dict):
        # keys are ids
        raw_ids = [str(k) for k in data.keys()]

    elif isinstance(data, list):
        for row in data:
            if isinstance(row, list) and row:
                raw_ids.append(str(row[0]))
            elif isinstance(row, dict):
                for cand in ("caseid", "case_id", "CASE_ID", "collision_id", "COLLISION_ID", "id", "ID"):
                    if cand in row:
                        raw_ids.append(str(row[cand]))
                        break
            else:
                # unknown row type; skip
                continue
    else:
        raise ValueError("'data' is neither dict nor list; cannot extract case IDs.")

    norm: List[str] = []
    seen = set()
    for r in raw_ids:
        n = normalize_case_id_for_download(r)
        if n and n not in seen:
            seen.add(n)
            norm.append(n)

    return norm


# =============================================================================
# Step 3: Download (download.php) and output plain CSV / optional Parquet
# =============================================================================
def write_parquet_parts_from_csv(
    src_csv: Path,
    out_dir: Path,
    part_prefix: str,
    chunksize: int = PARQUET_CSV_CHUNKSIZE,
) -> List[str]:
    """
    Convert CSV to parquet parts (per batch) using pandas chunks.
    Requires pandas + pyarrow.
    """
    import pandas as pd  # type: ignore
    import pyarrow  # noqa: F401

    ensure_dir(out_dir)
    written: List[str] = []
    part = 1
    for chunk in pd.read_csv(src_csv, chunksize=chunksize, low_memory=False):
        fn = f"{part_prefix}_part_{part:04d}.parquet"
        fp = out_dir / fn
        chunk.to_parquet(fp, index=False)
        written.append(str(fp))
        part += 1
    return written

def download_switrs_to_csv_or_parquet(
    sess: requests.Session,
    out_dir: Path,
    case_ids: List[str],
) -> Dict[str, Any]:
    """
    Download SWITRS data in batches.
    Output:
      - csv/<table>.csv (appended across batches)
      - parquet/<table>/batch_XXXX_part_YYYY.parquet (optional)

    We still may receive ZIP from TIMS, but we DO NOT keep ZIP files unless KEEP_RAW_DOWNLOADS=True.
    """
    url = "https://tims.berkeley.edu/tools/query/download.php"

    headers = {
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Content-Type": "application/x-www-form-urlencoded",
        "Origin": "https://tims.berkeley.edu",
        "Referer": "https://tims.berkeley.edu/tools/query/summary.php",
        "Upgrade-Insecure-Requests": "1",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome Safari",
    }

    downloads_dir = out_dir / "downloads"
    tmp_extract_dir = out_dir / "_tmp_extract"
    csv_out_dir = out_dir / "csv"
    pq_out_dir = out_dir / "parquet"

    ensure_dir(downloads_dir)
    ensure_dir(tmp_extract_dir)
    ensure_dir(csv_out_dir)
    if WRITE_PARQUET:
        ensure_dir(pq_out_dir)

    # Clean temp extract from previous runs
    if tmp_extract_dir.exists():
        shutil.rmtree(tmp_extract_dir, ignore_errors=True)
    ensure_dir(tmp_extract_dir)

    n_total = len(case_ids)
    batches = [case_ids[i:i + DOWNLOAD_BATCH_SIZE] for i in range(0, n_total, DOWNLOAD_BATCH_SIZE)]

    summary: Dict[str, Any] = {
        "endpoint": url,
        "case_count": n_total,
        "batch_size": DOWNLOAD_BATCH_SIZE,
        "num_batches": len(batches),
        "batches": [],
        "csv_outputs": [],
        "parquet_outputs": [],
    }

    for bi, batch in enumerate(batches, start=1):
        print(f"[DOWNLOAD] batch {bi}/{len(batches)}: {len(batch)} case IDs")

        # Save requested IDs per batch (helps confirm nothing got skipped)
        dump_json(downloads_dir / f"caseids_batch_{bi:04d}.json", {
            "batch": bi,
            "count": len(batch),
            "case_ids": batch,
        })

        form = {
            "caseid": json.dumps(batch),  # JSON array string
            "downloadType": "switrs",
            "selectedFilters": "",
        }

        resp = sess.post(url, headers=headers, data=form, timeout=DOWNLOAD_TIMEOUT_SEC)
        resp.raise_for_status()

        content = resp.content
        ct = (resp.headers.get("Content-Type") or "").lower()

        batch_info: Dict[str, Any] = {
            "batch": bi,
            "count": len(batch),
            "content_type": resp.headers.get("Content-Type"),
            "bytes": len(content),
            "status": "ok",
        }

        # HTML usually means an error/redirect page; keep it for inspection
        if "text/html" in ct:
            html_path = downloads_dir / f"switrs_batch_{bi:04d}.html"
            html_path.write_bytes(content)
            batch_info["status"] = "html_response"
            batch_info["saved"] = html_path.name
            summary["batches"].append(batch_info)
            continue

        # ZIP: extract CSVs and append to stable outputs
        if "zip" in ct or looks_like_zip(content):
            zip_path = downloads_dir / f"switrs_batch_{bi:04d}.zip"
            zip_path.write_bytes(content)

            extract_dir = tmp_extract_dir / f"batch_{bi:04d}"
            ensure_dir(extract_dir)

            try:
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(extract_dir)
            except zipfile.BadZipFile:
                batch_info["status"] = "bad_zip"
                batch_info["saved"] = zip_path.name
                summary["batches"].append(batch_info)
                continue

            csv_files = sorted(extract_dir.glob("**/*.csv"))
            batch_info["csv_files_in_zip"] = [str(p.relative_to(extract_dir)) for p in csv_files]

            for csvp in csv_files:
                dst = csv_out_dir / csvp.name
                append_csv_skip_header(csvp, dst)
                if str(dst) not in summary["csv_outputs"]:
                    summary["csv_outputs"].append(str(dst))

                if WRITE_PARQUET:
                    try:
                        table_dir = pq_out_dir / csvp.stem
                        written = write_parquet_parts_from_csv(
                            src_csv=csvp,
                            out_dir=table_dir,
                            part_prefix=f"batch_{bi:04d}",
                        )
                        summary["parquet_outputs"].extend(written)
                    except Exception as e:
                        batch_info.setdefault("parquet_errors", []).append({"csv": csvp.name, "error": str(e)})

            # Remove raw ZIP unless requested
            if not KEEP_RAW_DOWNLOADS:
                try:
                    zip_path.unlink()
                except Exception:
                    pass

            summary["batches"].append(batch_info)
            continue

        # Direct CSV response (less common for large requests)
        if "text/csv" in ct or (b"," in content[:2048]):
            raw_csv = downloads_dir / f"switrs_batch_{bi:04d}.csv"
            raw_csv.write_bytes(content)

            # Append into a single named CSV
            dst = csv_out_dir / "switrs.csv"
            append_csv_skip_header(raw_csv, dst)
            if str(dst) not in summary["csv_outputs"]:
                summary["csv_outputs"].append(str(dst))

            if WRITE_PARQUET:
                try:
                    table_dir = pq_out_dir / "switrs"
                    written = write_parquet_parts_from_csv(
                        src_csv=raw_csv,
                        out_dir=table_dir,
                        part_prefix=f"batch_{bi:04d}",
                    )
                    summary["parquet_outputs"].extend(written)
                except Exception as e:
                    batch_info.setdefault("parquet_errors", []).append({"csv": raw_csv.name, "error": str(e)})

            if not KEEP_RAW_DOWNLOADS:
                try:
                    raw_csv.unlink()
                except Exception:
                    pass

            batch_info["saved"] = "csv/switrs.csv"
            summary["batches"].append(batch_info)
            continue

        # Unknown binary: save for debugging
        bin_path = downloads_dir / f"switrs_batch_{bi:04d}.bin"
        bin_path.write_bytes(content)
        batch_info["status"] = "unknown_binary"
        batch_info["saved"] = bin_path.name
        summary["batches"].append(batch_info)

    # Clean temp extract
    if tmp_extract_dir.exists():
        shutil.rmtree(tmp_extract_dir, ignore_errors=True)

    # If we don't want raw downloads at all, but we did keep caseids_batch_*.json,
    # those are still useful for auditing; leave them.
    return summary


# =============================================================================
# Main
# =============================================================================
def main() -> None:
    print("TIMS SWITRS FULL PIPELINE: query -> response_full.json -> caseids -> download -> CSV/Parquet")
    print("-" * 90)

    start_date = validate_date_yyyy_mm_dd(input("Enter COLLISION start date (YYYY-MM-DD): "))
    end_date   = validate_date_yyyy_mm_dd(input("Enter COLLISION end date   (YYYY-MM-DD): "))
    county     = prompt_county()

    out_dir = make_output_dir_overwrite(OUT_ROOT, start_date, end_date, county)
    if CLEAR_PREVIOUS_RUN_ARTIFACTS:
        clear_run_artifacts(out_dir)
        # re-create subfolders when needed (folder itself remains)
        ensure_dir(out_dir)

    with requests.Session() as sess:
        # Step 1: query + save response_full.json
        payload, export_info = run_query_and_save(sess, out_dir, start_date, end_date, county)

        # Step 2: read the saved response_full.json (as you requested) and extract case IDs
        resp_path = out_dir / "response_full.json"
        case_ids = extract_case_ids_from_response_full_json(resp_path)
        print(f"\n[INFO] Extracted case IDs from response_full.json['data']: {len(case_ids)}")
        if case_ids:
            print("[INFO] First 20 case IDs:", case_ids[:20])
        else:
            print("[WARN] No case IDs extracted; download step will be skipped.")

        # Step 3: download and create CSV/Parquet outputs
        download_summary: Dict[str, Any]
        if case_ids:
            download_summary = download_switrs_to_csv_or_parquet(sess, out_dir, case_ids)
        else:
            download_summary = {"note": "No case IDs; download skipped."}

    # Manifest (overwrites each run)
    manifest = {
        "query_endpoint": "https://tims.berkeley.edu/tools/query/querySwitrsNew.php",
        "download_endpoint": "https://tims.berkeley.edu/tools/query/download.php",
        "request": {
            "start_date": start_date,
            "end_date": end_date,
            "county": county,
            "baseQuery": build_base_query(start_date, end_date, county),
            "pedorbike": "",
        },
        "config": {
            "DOWNLOAD_BATCH_SIZE": DOWNLOAD_BATCH_SIZE,
            "KEEP_RAW_DOWNLOADS": KEEP_RAW_DOWNLOADS,
            "WRITE_PARQUET": WRITE_PARQUET,
        },
        "query_export": export_info,
        "case_ids": {
            "count": len(case_ids),
            "example_first_20": case_ids[:20],
        },
        "download": download_summary,
        "saved_at_localtime": datetime.now().isoformat(timespec="seconds"),
    }
    dump_json(out_dir / "manifest.json", manifest)

    print("\n[SAVED] Output directory:")
    print(" ", out_dir.resolve())
    print("\n[SAVED] Key outputs:")
    print("  - response_full.json")
    print("  - manifest.json")
    print("  - csv/ (final CSV tables; appended across batches)")
    if WRITE_PARQUET:
        print("  - parquet/ (optional parquet parts)")
    print("  - downloads/ (per-batch requested case IDs; and any debug artifacts)")


if __name__ == "__main__":
    main()

TIMS SWITRS FULL PIPELINE: query -> response_full.json -> caseids -> download -> CSV/Parquet
------------------------------------------------------------------------------------------

Choose a COUNTY (no ALL). Options:
------------------------------------------------------------------------
 1) ALAMEDA           2) ALPINE            3) AMADOR            4) BUTTE            
 5) CALAVERAS         6) COLUSA            7) CONTRA COSTA      8) DEL NORTE        
 9) EL DORADO        10) FRESNO           11) GLENN            12) HUMBOLDT         
13) IMPERIAL         14) INYO             15) KERN             16) KINGS            
17) LAKE             18) LASSEN           19) LOS ANGELES      20) MADERA           
21) MARIN            22) MARIPOSA         23) MENDOCINO        24) MERCED           
25) MODOC            26) MONO             27) MONTEREY         28) NAPA             
29) NEVADA           30) ORANGE           31) PLACER           32) PLUMAS           
33) RIVERSIDE        34) SA

In [24]:
#!/usr/bin/env python3
"""
TIMS SWITRS pipeline (STATEWIDE, no prompts)
===========================================

Change from your original:
- NO user input prompts
- Iterates ALL counties in COUNTIES
- Uses fixed date range: 2000-01-01 to 2023-12-31
- For each county: runs the same 2-step pipeline and writes outputs under:
    out_tims_json/2000-01-01_to_2023-12-31_<COUNTY>/

Notes:
- This will generate ONE folder per county (same structure as your current script).
- If you want ONE single merged CSV across all counties, say so and I’ll modify the
  write/append logic (it’s a different aggregation step).

Run:
  python tims_switrs_statewide_by_county.py
"""

from __future__ import annotations

import json
import re
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests


# =============================================================================
# USER CONFIG
# =============================================================================
OUT_ROOT = Path("out_tims_json")

# Fixed statewide time range
START_DATE = "2000-01-01"
END_DATE   = "2023-12-31"

# Query response saving (usually dict; chunking is rarely needed)
QUERY_CHUNK_SIZE = 50_000  # only used if payload is a huge list

# Download batching (smaller is safer if the server truncates silently)
DOWNLOAD_BATCH_SIZE = 500      # try 200-1000 if you see issues
DOWNLOAD_TIMEOUT_SEC = 300

# What to output
KEEP_RAW_DOWNLOADS = False     # if True, keeps raw zip/csv/html from server in downloads/
WRITE_PARQUET = False          # if True, writes parquet parts under parquet/ (needs pandas+pyarrow)
PARQUET_CSV_CHUNKSIZE = 200_000

# Overwrite semantics inside existing folder
CLEAR_PREVIOUS_RUN_ARTIFACTS = True  # clears csv/, parquet/, downloads/, temp extract each run


# =============================================================================
# County list (NO "ALL")
# =============================================================================
COUNTIES = [
    "ALAMEDA","ALPINE","AMADOR","BUTTE","CALAVERAS","COLUSA","CONTRA COSTA","DEL NORTE",
    "EL DORADO","FRESNO","GLENN","HUMBOLDT","IMPERIAL","INYO","KERN","KINGS","LAKE","LASSEN",
    "LOS ANGELES","MADERA","MARIN","MARIPOSA","MENDOCINO","MERCED","MODOC","MONO","MONTEREY","NAPA",
    "NEVADA","ORANGE","PLACER","PLUMAS","RIVERSIDE","SACRAMENTO","SAN BENITO","SAN BERNARDINO",
    "SAN DIEGO","SAN FRANCISCO","SAN JOAQUIN","SAN LUIS OBISPO","SAN MATEO","SANTA BARBARA",
    "SANTA CLARA","SANTA CRUZ","SHASTA","SIERRA","SISKIYOU","SOLANO","SONOMA","STANISLAUS","SUTTER",
    "TEHAMA","TRINITY","TULARE","TUOLUMNE","VENTURA","YOLO","YUBA"
]


# =============================================================================
# Helpers: formatting
# =============================================================================
def validate_date_yyyy_mm_dd(s: str) -> str:
    s = s.strip()
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        raise ValueError(f"Date must be YYYY-MM-DD, got: {s}")
    return s

def build_base_query(start_date: str, end_date: str, county: str) -> str:
    return (
        f"COLLISION_DATE >= '{start_date}' "
        f"AND COLLISION_DATE <= '{end_date}' "
        f"AND S.COUNTY = '{county}'"
    )

def safe_tag(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[^A-Za-z0-9._-]+", "-", s)
    return s.strip("-")


# =============================================================================
# Helpers: filesystem
# =============================================================================
def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def dump_json(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def make_output_dir_overwrite(out_root: Path, start_date: str, end_date: str, county: str) -> Path:
    """
    Deterministic folder, reuse it; overwrite contents.
    """
    out_root.mkdir(parents=True, exist_ok=True)
    out_dir = out_root / safe_tag(f"{start_date}_to_{end_date}_{county}")
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

def clear_run_artifacts(out_dir: Path) -> None:
    """
    Clears subfolders and known files, but does NOT delete/recreate the main folder.
    """
    for sub in ["csv", "parquet", "downloads", "_tmp_extract"]:
        p = out_dir / sub
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)

    for fn in ["response_full.json", "manifest.json", "response_raw_text.txt"]:
        p = out_dir / fn
        if p.exists():
            try:
                p.unlink()
            except Exception:
                pass

def looks_like_zip(content: bytes) -> bool:
    return len(content) >= 4 and content[:2] == b"PK"  # ZIP magic

def append_csv_skip_header(src_csv: Path, dst_csv: Path) -> None:
    """
    Append src_csv to dst_csv (streaming).
    If dst doesn't exist: copy whole file.
    If dst exists: append everything except first line (header).
    """
    ensure_dir(dst_csv.parent)
    if not dst_csv.exists():
        shutil.copyfile(src_csv, dst_csv)
        return

    with src_csv.open("r", encoding="utf-8", errors="replace") as fin, \
         dst_csv.open("a", encoding="utf-8", errors="replace") as fout:
        _ = fin.readline()  # skip header
        shutil.copyfileobj(fin, fout)


# =============================================================================
# Step 1: Query + save response_full.json
# =============================================================================
def find_primary_list_container(payload: Any) -> Optional[str]:
    if not isinstance(payload, dict):
        return None
    for key in ("data", "results", "rows"):
        if key in payload and isinstance(payload[key], list):
            return key
    return None

def export_query_payload(out_dir: Path, payload: Any) -> Dict[str, Any]:
    """
    Save query payload as response_full.json (or chunked).
    """
    for p in out_dir.glob("response_chunk_*.json"):
        try:
            p.unlink()
        except Exception:
            pass

    export_info: Dict[str, Any] = {"mode": None, "files": []}

    if isinstance(payload, list):
        n = len(payload)
        if n > QUERY_CHUNK_SIZE:
            export_info["mode"] = "chunked_list"
            num_chunks = (n + QUERY_CHUNK_SIZE - 1) // QUERY_CHUNK_SIZE
            for i in range(num_chunks):
                chunk = payload[i * QUERY_CHUNK_SIZE : (i + 1) * QUERY_CHUNK_SIZE]
                fn = f"response_chunk_{i+1:04d}.json"
                dump_json(out_dir / fn, chunk)
                export_info["files"].append(fn)
            export_info["total_rows"] = n
            export_info["chunk_size"] = QUERY_CHUNK_SIZE
        else:
            export_info["mode"] = "single"
            dump_json(out_dir / "response_full.json", payload)
            export_info["files"].append("response_full.json")
            export_info["total_rows"] = n
        return export_info

    primary = find_primary_list_container(payload)
    if primary is not None and isinstance(payload.get(primary), list):
        export_info["container_key"] = primary
        export_info["total_rows"] = len(payload[primary])

    export_info["mode"] = "single"
    dump_json(out_dir / "response_full.json", payload)
    export_info["files"].append("response_full.json")
    return export_info

def run_query_and_save(
    sess: requests.Session,
    out_dir: Path,
    start_date: str,
    end_date: str,
    county: str,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    url = "https://tims.berkeley.edu/tools/query/querySwitrsNew.php"
    base_query = build_base_query(start_date, end_date, county)
    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Origin": "https://tims.berkeley.edu",
        "Referer": "https://tims.berkeley.edu/tools/query/summary.php",
        "X-Requested-With": "XMLHttpRequest",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome Safari",
    }
    data = {"baseQuery": base_query, "pedorbike": ""}

    print("\n[INFO] POSTing query to:", url)
    print("[INFO] baseQuery:", base_query)

    resp = sess.post(url, headers=headers, data=data, timeout=120)
    resp.raise_for_status()

    try:
        payload = resp.json()
    except Exception:
        raw_path = out_dir / "response_raw_text.txt"
        raw_path.write_text(resp.text, encoding="utf-8")
        raise RuntimeError(f"Query response not valid JSON. Saved raw text to: {raw_path}")

    export_info = export_query_payload(out_dir, payload)
    return payload, export_info


# =============================================================================
# Step 2: Extract case IDs (keep exact, only strip leading '22' for 9-digit pattern)
# =============================================================================
def normalize_case_id_for_download(raw_id: str) -> Optional[str]:
    s = str(raw_id).strip().replace("\ufeff", "")
    if not s.isdigit():
        return None
    m = re.fullmatch(r"22(\d{7})", s)
    if m:
        return m.group(1)
    return s

def extract_case_ids_from_response_full_json(path: Path) -> List[str]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict) or "data" not in payload:
        raise ValueError("response_full.json does not have top-level 'data' key (or not a dict).")

    data = payload["data"]
    raw_ids: List[str] = []

    if isinstance(data, dict):
        raw_ids = [str(k) for k in data.keys()]
    elif isinstance(data, list):
        for row in data:
            if isinstance(row, list) and row:
                raw_ids.append(str(row[0]))
            elif isinstance(row, dict):
                for cand in ("caseid", "case_id", "CASE_ID", "collision_id", "COLLISION_ID", "id", "ID"):
                    if cand in row:
                        raw_ids.append(str(row[cand]))
                        break
    else:
        raise ValueError("'data' is neither dict nor list; cannot extract case IDs.")

    norm: List[str] = []
    seen = set()
    for r in raw_ids:
        n = normalize_case_id_for_download(r)
        if n and n not in seen:
            seen.add(n)
            norm.append(n)
    return norm


# =============================================================================
# Step 3: Download and output plain CSV / optional Parquet (unchanged)
# =============================================================================
def write_parquet_parts_from_csv(
    src_csv: Path,
    out_dir: Path,
    part_prefix: str,
    chunksize: int = PARQUET_CSV_CHUNKSIZE,
) -> List[str]:
    import pandas as pd  # type: ignore
    import pyarrow  # noqa: F401

    ensure_dir(out_dir)
    written: List[str] = []
    part = 1
    for chunk in pd.read_csv(src_csv, chunksize=chunksize, low_memory=False):
        fn = f"{part_prefix}_part_{part:04d}.parquet"
        fp = out_dir / fn
        chunk.to_parquet(fp, index=False)
        written.append(str(fp))
        part += 1
    return written

def download_switrs_to_csv_or_parquet(
    sess: requests.Session,
    out_dir: Path,
    case_ids: List[str],
) -> Dict[str, Any]:
    url = "https://tims.berkeley.edu/tools/query/download.php"

    headers = {
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Content-Type": "application/x-www-form-urlencoded",
        "Origin": "https://tims.berkeley.edu",
        "Referer": "https://tims.berkeley.edu/tools/query/summary.php",
        "Upgrade-Insecure-Requests": "1",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome Safari",
    }

    downloads_dir = out_dir / "downloads"
    tmp_extract_dir = out_dir / "_tmp_extract"
    csv_out_dir = out_dir / "csv"
    pq_out_dir = out_dir / "parquet"

    ensure_dir(downloads_dir)
    ensure_dir(tmp_extract_dir)
    ensure_dir(csv_out_dir)
    if WRITE_PARQUET:
        ensure_dir(pq_out_dir)

    if tmp_extract_dir.exists():
        shutil.rmtree(tmp_extract_dir, ignore_errors=True)
    ensure_dir(tmp_extract_dir)

    n_total = len(case_ids)
    batches = [case_ids[i:i + DOWNLOAD_BATCH_SIZE] for i in range(0, n_total, DOWNLOAD_BATCH_SIZE)]

    summary: Dict[str, Any] = {
        "endpoint": url,
        "case_count": n_total,
        "batch_size": DOWNLOAD_BATCH_SIZE,
        "num_batches": len(batches),
        "batches": [],
        "csv_outputs": [],
        "parquet_outputs": [],
    }

    for bi, batch in enumerate(batches, start=1):
        print(f"[DOWNLOAD] batch {bi}/{len(batches)}: {len(batch)} case IDs")

        dump_json(downloads_dir / f"caseids_batch_{bi:04d}.json", {
            "batch": bi, "count": len(batch), "case_ids": batch,
        })

        form = {
            "caseid": json.dumps(batch),
            "downloadType": "switrs",
            "selectedFilters": "",
        }

        resp = sess.post(url, headers=headers, data=form, timeout=DOWNLOAD_TIMEOUT_SEC)
        resp.raise_for_status()

        content = resp.content
        ct = (resp.headers.get("Content-Type") or "").lower()

        batch_info: Dict[str, Any] = {
            "batch": bi,
            "count": len(batch),
            "content_type": resp.headers.get("Content-Type"),
            "bytes": len(content),
            "status": "ok",
        }

        if "text/html" in ct:
            html_path = downloads_dir / f"switrs_batch_{bi:04d}.html"
            html_path.write_bytes(content)
            batch_info["status"] = "html_response"
            batch_info["saved"] = html_path.name
            summary["batches"].append(batch_info)
            continue

        if "zip" in ct or looks_like_zip(content):
            zip_path = downloads_dir / f"switrs_batch_{bi:04d}.zip"
            zip_path.write_bytes(content)

            extract_dir = tmp_extract_dir / f"batch_{bi:04d}"
            ensure_dir(extract_dir)

            try:
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(extract_dir)
            except zipfile.BadZipFile:
                batch_info["status"] = "bad_zip"
                batch_info["saved"] = zip_path.name
                summary["batches"].append(batch_info)
                continue

            csv_files = sorted(extract_dir.glob("**/*.csv"))
            batch_info["csv_files_in_zip"] = [str(p.relative_to(extract_dir)) for p in csv_files]

            for csvp in csv_files:
                dst = csv_out_dir / csvp.name
                append_csv_skip_header(csvp, dst)
                if str(dst) not in summary["csv_outputs"]:
                    summary["csv_outputs"].append(str(dst))

                if WRITE_PARQUET:
                    try:
                        table_dir = pq_out_dir / csvp.stem
                        written = write_parquet_parts_from_csv(
                            src_csv=csvp,
                            out_dir=table_dir,
                            part_prefix=f"batch_{bi:04d}",
                        )
                        summary["parquet_outputs"].extend(written)
                    except Exception as e:
                        batch_info.setdefault("parquet_errors", []).append({"csv": csvp.name, "error": str(e)})

            if not KEEP_RAW_DOWNLOADS:
                try:
                    zip_path.unlink()
                except Exception:
                    pass

            summary["batches"].append(batch_info)
            continue

        if "text/csv" in ct or (b"," in content[:2048]):
            raw_csv = downloads_dir / f"switrs_batch_{bi:04d}.csv"
            raw_csv.write_bytes(content)

            dst = csv_out_dir / "switrs.csv"
            append_csv_skip_header(raw_csv, dst)
            if str(dst) not in summary["csv_outputs"]:
                summary["csv_outputs"].append(str(dst))

            if WRITE_PARQUET:
                try:
                    table_dir = pq_out_dir / "switrs"
                    written = write_parquet_parts_from_csv(
                        src_csv=raw_csv,
                        out_dir=table_dir,
                        part_prefix=f"batch_{bi:04d}",
                    )
                    summary["parquet_outputs"].extend(written)
                except Exception as e:
                    batch_info.setdefault("parquet_errors", []).append({"csv": raw_csv.name, "error": str(e)})

            if not KEEP_RAW_DOWNLOADS:
                try:
                    raw_csv.unlink()
                except Exception:
                    pass

            batch_info["saved"] = "csv/switrs.csv"
            summary["batches"].append(batch_info)
            continue

        bin_path = downloads_dir / f"switrs_batch_{bi:04d}.bin"
        bin_path.write_bytes(content)
        batch_info["status"] = "unknown_binary"
        batch_info["saved"] = bin_path.name
        summary["batches"].append(batch_info)

    if tmp_extract_dir.exists():
        shutil.rmtree(tmp_extract_dir, ignore_errors=True)

    return summary


# =============================================================================
# Main (STATEWIDE: no prompts)
# =============================================================================
def main() -> None:
    print("TIMS SWITRS STATEWIDE PIPELINE: for each county -> query -> caseids -> download -> CSV/Parquet")
    print("-" * 90)

    start_date = validate_date_yyyy_mm_dd(START_DATE)
    end_date = validate_date_yyyy_mm_dd(END_DATE)

    with requests.Session() as sess:
        for i, county in enumerate(COUNTIES, start=1):
            print("\n" + "=" * 90)
            print(f"[COUNTY] {i}/{len(COUNTIES)}: {county}")
            print("=" * 90)

            out_dir = make_output_dir_overwrite(OUT_ROOT, start_date, end_date, county)
            if CLEAR_PREVIOUS_RUN_ARTIFACTS:
                clear_run_artifacts(out_dir)
                ensure_dir(out_dir)

            # Step 1: query + save response_full.json
            payload, export_info = run_query_and_save(sess, out_dir, start_date, end_date, county)

            # Step 2: extract case IDs
            resp_path = out_dir / "response_full.json"
            case_ids = extract_case_ids_from_response_full_json(resp_path)
            print(f"[INFO] Extracted case IDs: {len(case_ids)}")
            if case_ids:
                print("[INFO] First 20 case IDs:", case_ids[:20])
            else:
                print("[WARN] No case IDs extracted; download will be skipped.")

            # Step 3: download
            if case_ids:
                download_summary = download_switrs_to_csv_or_parquet(sess, out_dir, case_ids)
            else:
                download_summary = {"note": "No case IDs; download skipped."}

            # Manifest per county
            manifest = {
                "query_endpoint": "https://tims.berkeley.edu/tools/query/querySwitrsNew.php",
                "download_endpoint": "https://tims.berkeley.edu/tools/query/download.php",
                "request": {
                    "start_date": start_date,
                    "end_date": end_date,
                    "county": county,
                    "baseQuery": build_base_query(start_date, end_date, county),
                    "pedorbike": "",
                },
                "config": {
                    "DOWNLOAD_BATCH_SIZE": DOWNLOAD_BATCH_SIZE,
                    "KEEP_RAW_DOWNLOADS": KEEP_RAW_DOWNLOADS,
                    "WRITE_PARQUET": WRITE_PARQUET,
                },
                "query_export": export_info,
                "case_ids": {
                    "count": len(case_ids),
                    "example_first_20": case_ids[:20],
                },
                "download": download_summary,
                "saved_at_localtime": datetime.now().isoformat(timespec="seconds"),
            }
            dump_json(out_dir / "manifest.json", manifest)

            print("[SAVED] ", out_dir.resolve())

    print("\n[DONE] Finished all counties.")


if __name__ == "__main__":
    main()

TIMS SWITRS STATEWIDE PIPELINE: for each county -> query -> caseids -> download -> CSV/Parquet
------------------------------------------------------------------------------------------

[COUNTY] 1/58: ALAMEDA

[INFO] POSTing query to: https://tims.berkeley.edu/tools/query/querySwitrsNew.php
[INFO] baseQuery: COLLISION_DATE >= '2000-01-01' AND COLLISION_DATE <= '2023-12-31' AND S.COUNTY = 'ALAMEDA'
[INFO] Extracted case IDs: 180835
[INFO] First 20 case IDs: ['0000130', '0000131', '0000132', '0000133', '0000188', '0000323', '0000338', '0000344', '0000345', '0000557', '0000737', '0000738', '0000739', '0000906', '0000913', '0000915', '0001461', '0001465', '0001470', '0001476']
[DOWNLOAD] batch 1/362: 500 case IDs
[DOWNLOAD] batch 2/362: 500 case IDs
[DOWNLOAD] batch 3/362: 500 case IDs
[DOWNLOAD] batch 4/362: 500 case IDs
[DOWNLOAD] batch 5/362: 500 case IDs
[DOWNLOAD] batch 6/362: 500 case IDs
[DOWNLOAD] batch 7/362: 500 case IDs
[DOWNLOAD] batch 8/362: 500 case IDs
[DOWNLOAD] batch 9/36

KeyboardInterrupt: 